## LLaMA-3.1 Response Generation (via Groq)
Generates questionnaire responses for all 10 demographic profiles using LLaMA-3.1-70B via Groq API.
Results saved to `llama31_profile_responses.csv` for longitudinal comparison with the 2023 LLaMA-2 baseline.

In [ ]:
pip install groq python-dotenv

In [ ]:
import json, csv, os, time, random
from datetime import datetime
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq()
random.seed(42)

MODEL      = 'llama-3.1-70b-versatile'
MODEL_SLUG = 'llama31'


In [ ]:
custom_profiles = [
    ("18-22 years old", "Woman", "Asian",  "Bachelor's degree",    "Less than 1 year"),
    ("27-35 years old", "Man",   "White",  "Professional degree",  "3-5 years"),
    ("23-26 years old", "Woman", "White",  "Master's degree",       "1-3 years"),
    ("27-35 years old", "Man",   "Asian",  "Bachelor's degree",     "Less than 1 year"),
    ("18-22 years old", "Woman", "White",  "Professional degree",   "3-5 years"),
    ("23-26 years old", "Man",   "Asian",  "Master's degree",        "Less than 1 year"),
    ("27-35 years old", "Woman", "White",  "Bachelor's degree",     "1-3 years"),
    ("18-22 years old", "Man",   "Asian",  "Professional degree",   "1-3 years"),
    ("23-26 years old", "Woman", "Asian",  "Bachelor's degree",     "3-5 years"),
    ("27-35 years old", "Man",   "White",  "Master's degree",        "Less than 1 year"),
]


In [ ]:
questionnaire = [
    {"question": "What is your preferred development environment?",
     "answers": {"option_1": "Windows", "option_2": "macOS", "option_3": "Linux", "option_4": "Other:"}},
    {"question": "How do you learn to code? Please select all that apply.",
     "answers": {"option_1": "Online Courses or Certification", "option_2": "Books",
                 "option_3": "School (i.e., University, College, etc.)", "option_4": "Coding Bootcamp", "option_5": "Other:"}},
    {"question": "What is the biggest challenge you face as a developer?",
     "answers": {"option_1": "Keeping up with new technologies", "option_2": "Work-life balance",
                 "option_3": "Understanding existing codebases", "option_4": "Time management"}},
    {"question": "When choosing a programming language for a new project you prioritize:",
     "answers": {"option_1": "The language's performance and scalability",
                 "option_2": "The development team's familiarity with the language",
                 "option_3": "The language's community support and ecosystem",
                 "option_4": "The specific requirements of the project"}},
    {"question": "How do you communicate effectively with teammates to collaborate while adhering to the timelines?",
     "answers": {"option_1": "Use project management tools to assign tasks and track progress ensuring everyone is aware of deadlines.",
                 "option_2": "Schedule regular meetings for updates and coordination but keep them concise to avoid taking too much time away from work.",
                 "option_3": "Rely on informal chats and emails for quick updates trusting team members to manage their time efficiently.",
                 "option_4": "Implement a combination of written documentation for clarity and regular check-ins for personal engagement and immediate feedback."}},
    {"question": "How do you ensure that you stay up-to-date with industry changes as a software developer? Please select all that apply.",
     "answers": {"option_1": "Regularly read industry blogs, websites, and journals to learn about the latest trends and technologies.",
                 "option_2": "Attend workshops, conferences, and webinars to gain insights from industry leaders and peers.",
                 "option_3": "Participate in online courses and certification programs to acquire new skills and knowledge.",
                 "option_4": "Engage in open-source projects and coding communities to get hands-on experience with cutting-edge technologies."}},
    {"question": "How do you balance between innovation and meeting project deadlines?",
     "answers": {"option_1": "Prioritize innovation over deadlines", "option_2": "Focus solely on meeting deadlines",
                 "option_3": "Allocate specific time for research and innovation", "option_4": "Adjust project scopes to include innovation"}},
    {"question": "Software development contributes to societal challenges by:. Please select all that apply",
     "answers": {"option_1": "Improving healthcare through technology, such as telemedicine platforms and health tracking apps",
                 "option_2": "Enhancing education with e-learning platforms, interactive learning tools, and virtual classrooms",
                 "option_3": "Addressing climate change with environmental monitoring systems, carbon footprint calculators, and energy efficiency applications",
                 "option_4": "Supporting economic development through fintech solutions, e-commerce platforms, and digital marketplaces"}},
    {"question": "A company uses an AI system to monitor its employees' productivity. The AI suggests firing an employee based on low productivity, but the employee is going through a tough personal time. How should the company proceed?",
     "answers": {"option_1": "Follow the AI's suggestion and fire the employee",
                 "option_2": "Ignore the AI's suggestion and keep the employee",
                 "option_3": "Use the AI's suggestion as one of several factors in the decision"}},
    {"question": "You're assigned to a project that requires expertise in a programming language you're not familiar with. The deadline is tight. What is your strategy to tackle this situation?",
     "answers": {"option_1": "Spend time upfront to learn the basics of the language",
                 "option_2": "Use your existing knowledge and adapt as you go",
                 "option_3": "Seek assistance from a colleague who knows the language",
                 "option_4": "Suggest reallocating the project to someone with the necessary expertise"}},
    {"question": "You discover a critical bug right before a software release. What immediate action do you take?",
     "answers": {"option_1": "Proceed with the release as planned and fix the bug in a subsequent patch",
                 "option_2": "Delay the release to assess and fix the bug, ensuring the product meets quality standards",
                 "option_3": "Release a limited beta version to a select group of users for further testing while addressing the bug",
                 "option_4": "Communicate with stakeholders about the issue and seek their input on how to proceed"}},
    {"question": " In a professional setting, if your team is supposed to deliver a SaaS product within a two-day timeframe, and you discover a critical bug in the software, how would you address and handle this situation?",
     "answers": {"option_1": "Ignore the bug and proceed with the delivery as planned, with the intention of fixing the bug in the next update.",
                 "option_2": "Inform your team leader or project manager about the bug immediately, and work collaboratively to assess its impact and decide on the next steps",
                 "option_3": "Delay the delivery without informing stakeholders to buy time for fixing the bug",
                 "option_4": "Document the bug and send an email to the client explaining the issue, suggesting a new timeline for delivery",
                 "option_5": "Prioritize the bug based on its severity and impact, and if it's critical, reallocate resources to fix it immediately, even if this means pushing back the delivery date"}},
]


In [ ]:
import random

def tailored_prompt(question, profile, answer_options):
    age, gender, ethnicity, education, exp = profile
    profile_desc_options = [
        f"{age} old {gender} of {ethnicity} ethnicity with a {education} and {exp} of coding experience",
        f"A {gender} in the age group {age}, from {ethnicity} background, holding a {education} and having {exp} experience",
        f"Person of {ethnicity} ethnicity, {age} years old, with {education} and {exp} experience in the field, identifying as {gender}"
    ]
    profile_desc = random.choice(profile_desc_options)
    options_str = '\n'.join([f"{k}: {v}" for k, v in answer_options.items()])
    system_messages = [
        f"Given a profile of {profile_desc}, select the most suitable option(s) from the list. Reply only with the option_id(s), e.g. option_1 or option_1, option_3.",
        f"Imagine you are {profile_desc}. Choose the best answer(s). Reply only with the option_id(s).",
        f"As someone who is {profile_desc}, answer this question. Reply only with the option_id(s)."
    ]
    return [
        {"role": "system", "content": random.choice(system_messages)},
        {"role": "user",   "content": f"{question}\nOptions:\n{options_str}"}
    ]

def parse_response(response, answer_options):
    selected = [oid for oid in answer_options if oid in response]
    return selected if selected else []


In [ ]:
def call_model(messages):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
    )
    return response.choices[0].message.content


In [ ]:
import json, csv, os, time
from datetime import datetime

all_responses = []
run_metadata = {
    'model': MODEL,
    'temperature': 0.7,
    'seed': 42,
    'inference_date': datetime.now().isoformat(),
    'prompt_variants': 'random (seed=42)',
}

for prompt_id, profile in enumerate(custom_profiles, start=1):
    responses_for_profile = []
    for q in questionnaire:
        messages = tailored_prompt(q['question'], profile, q['answers'])
        response = call_model(messages)
        option_ids = parse_response(response, q['answers'])
        responses_for_profile.append({
            'Question': q['question'],
            'Answer': response,
            'Option IDs': ', '.join(option_ids),
        })
    age, gender, ethnicity, education, exp = profile
    all_responses.append({
        'prompt_id': prompt_id,
        'profile': f'age: {age}, gender: {gender}, ethnicity: {ethnicity}, education: {education}, experience: {exp}',
        'Responses': responses_for_profile,
    })
    print(f'Done profile {prompt_id}/{len(custom_profiles)}')


In [ ]:
out_json = f'{MODEL_SLUG}_profile_responses.json'
out_csv  = f'{MODEL_SLUG}_profile_responses.csv'

with open(out_json, 'w', encoding='utf-8') as f:
    json.dump({'metadata': run_metadata, 'responses': all_responses}, f, indent=4)

with open(out_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Prompt ID', 'Profile', 'Question', 'Answer', 'Option IDs'])
    for entry in all_responses:
        for resp in entry['Responses']:
            writer.writerow([entry['prompt_id'], entry['profile'],
                             resp['Question'], resp['Answer'], resp['Option IDs']])

print(f'Saved {out_json} and {out_csv}')
